# DPO for spoken style — procedure

Spike 02 of `research/`. **This notebook is checked in unexecuted.** It records the
procedure a real run would follow. The claim, the falsifier and the outcome are in
`README.md` beside it.

Cells marked **light** run against `vmp` with the standard library only. Cells marked
**heavy** need `torch`, `transformers`, `peft`, `trl` and `datasets`.

| Step | Section |
|---|---|
| 1 | Corpus — prompts and candidate replies |
| 2 | Pairs — the rule that labels them |
| 3 | Baseline — the prompt-only control |
| 4 | Train — DPO on the base model, no SFT stage |
| 5 | Eval — violation rate and correctness together |
| 6 | Caveats |

## 1. Corpus

Prompts come from the synthetic golden set so the same prompts can be reused by the
evaluation later. Candidates are generated three ways: a short spoken reply, a
markdown-formatted reply, and a long multi-paragraph reply. In a real run the candidates
come from the base model at different temperatures and system prompts — what matters is
that several replies to the same prompt differ in style.

In [ ]:
# light
import sys

sys.path.insert(0, "../../src")

from vmp.data.synthetic import generate_golden_set

CATEGORIES = ("factual_short", "tool_intent", "instruction_format")


def build_replies(utterance) -> list[str]:
    """Three candidates for one prompt: spoken, markdown, multi-paragraph."""
    answer = utterance.meta.get("answer") or "the requested action"
    return [
        f"It's {answer}. Let me know if you want the longer explanation.",
        f"## Answer\n- **Result:** {answer}\n- See `docs/index.md` for the details\n",
        (
            f"That is a good question, and the short version is {answer}. Before getting "
            "to it, some background, because the answer depends on assumptions that are "
            "easy to miss and that people often disagree about in practice.\n\n"
            "With that in place, the reasoning runs as follows. Several conventions are in "
            "common use, they mostly agree, and where they differ the difference rarely "
            "matters for everyday purposes."
        ),
    ]


prompts = generate_golden_set(per_category=6, seed=7, categories=CATEGORIES)
records = [
    {"prompt": u.text, "category": u.meta["category"], "replies": build_replies(u)}
    for u in prompts
]
expected = [str(u.meta.get("answer", "")) for u in prompts]
len(records)

## 2. Pairs

The label is a function, not a person. `spoken_style_violations` returns the rules a
reply breaks; an empty list makes it a `chosen`, a non-empty list makes it a `rejected`
for the same prompt. `pairs_from_style` emits every (clean, violating) combination.

Check the statistics before training. If `chosen_shorter_fraction` is near 1.0 the
dataset may be teaching brevity rather than spoken style — the confound the falsifier's
second clause is written against.

In [ ]:
# light
from vmp.data.corpus import PreferencePairBuilder, spoken_style_violations
from vmp.training.dpo import preference_stats
from vmp.training.plan import write_jsonl

pairs = PreferencePairBuilder(source="rule:spoken-style").build(records)
write_jsonl("data/preference_pairs.jsonl", [p.to_dict() for p in pairs])

preference_stats(pairs)

## 3. Baseline — the control that matters

Before training anything, measure two baselines on **held-out** prompts:

1. the base model with a neutral system prompt;
2. the base model with the spoken-style system prompt
   (`vmp.training.sft.SPOKEN_SYSTEM_PROMPT`).

If (2) already removes the violations, the experiment is over and the answer is "use a
prompt". Skipping this cell is how a training run takes credit for what an instruction
would have done.

In [ ]:
# heavy — generation with the base model
from transformers import pipeline

from vmp.training.sft import SPOKEN_SYSTEM_PROMPT

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
chat = pipeline("text-generation", model=BASE_MODEL)


def generate(prompt: str, system: str, adapter: str | None = None) -> str:
    pipe = chat if adapter is None else pipeline("text-generation", model=adapter)
    messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
    out = pipe(messages, max_new_tokens=256, do_sample=False)
    return out[0]["generated_text"][-1]["content"].strip()


held_out_utterances = generate_golden_set(per_category=6, seed=11, categories=CATEGORIES)
held_out = [u.text for u in held_out_utterances]
held_out_expected = [str(u.meta.get("answer", "")) for u in held_out_utterances]


def violation_rate(system_prompt: str, prompts: list[str]) -> float:
    replies = [generate(p, system=system_prompt) for p in prompts]
    return sum(bool(spoken_style_violations(r)) for r in replies) / len(replies)


print("neutral prompt :", violation_rate("You are a helpful assistant.", held_out))
print("spoken prompt  :", violation_rate(SPOKEN_SYSTEM_PROMPT, held_out))

## 4. Train

The plan is `configs/train_dpo.toml`, loaded as a `TrainingPlan` so the notebook and the
package train the same thing. `ref_model=None`: the reference policy is the same weights
with the LoRA adapter disabled, so no SFT checkpoint and no second copy of the base model
is needed.

Run the dry run first — it validates the pair file, hashes it and estimates the steps
without importing anything heavy.

In [ ]:
# light
from vmp.training.dpo import run_dpo
from vmp.training.plan import TrainingPlan

plan = TrainingPlan.from_toml("../../configs/train_dpo.toml")
plan.datasets = {"train": "data/preference_pairs.jsonl"}

manifest = run_dpo(plan, dry_run=True)
manifest["estimated_steps"], manifest["beta"], manifest["pairs"]["n"], manifest["warnings"]

In [ ]:
# heavy — TRL DPOTrainer with a PEFT LoRA adapter; writes to plan.output_dir
result = run_dpo(plan, dry_run=False)
result["result"]

## 5. Eval

Two metrics, reported together, on the held-out prompts from step 3:

- **violation rate** — the fraction of replies breaking at least one spoken-style rule.
  This is the metric the training optimised, so it is expected to move; it is evidence
  the training worked, not that it helped.
- **correctness** — exact match against the golden set's expected answers, via
  `vmp.eval.wer.exact_match`. This is the metric that can silently pay for the first one,
  and the falsifier's second clause is written against it.

`win_rate` gives a third view: how often the adapted reply is the rule-preferred one when
placed beside the base model's reply.

In [ ]:
# heavy for generation, light for scoring
from vmp.eval.judge import RubricJudge
from vmp.eval.preference import win_rate
from vmp.eval.wer import exact_match
from vmp.types import PreferencePair

base_replies = [generate(p, SPOKEN_SYSTEM_PROMPT, adapter=None) for p in held_out]
dpo_replies = [generate(p, SPOKEN_SYSTEM_PROMPT, adapter=plan.output_dir) for p in held_out]

for name, replies in (("base", base_replies), ("dpo", dpo_replies)):
    violations = sum(bool(spoken_style_violations(r)) for r in replies) / len(replies)
    correct = sum(
        exact_match(exp, reply, numbers=True)
        for exp, reply in zip(held_out_expected, replies, strict=True)
    ) / len(replies)
    print(f"{name:<5} violation_rate {violations:.3f}   exact_match {correct:.3f}")

# The reference each policy answer must beat is the base model's own reply. Only
# `chosen` is read when reference="chosen", which is the default.
comparison = [
    PreferencePair(prompt=prompt, chosen=base, rejected=base, source="base-model")
    for prompt, base in zip(held_out, base_replies, strict=True)
]
win_rate(comparison, dpo_replies, judge=RubricJudge())

## 6. Caveats

- **The metric is the training signal.** The violation rate is computed by the same
  function that labelled the pairs. A model that learns to satisfy the rule has not
  necessarily learned to speak well, and this metric cannot tell the difference. A rubric
  or LLM judge (`vmp.eval.judge`) is the check on the check.
- **Brevity is the confound.** Every rejected reply here is also longer. Pairs where
  chosen and rejected have the same word count — markdown versus plain — separate style
  from length and belong in any serious dataset.
- **The prompt-only control is not optional.** See step 3.
- **No held-out prompts, no result.** The pairs are derived from prompts; evaluating on
  those same prompts measures memorisation.
- **`beta` is a real knob.** A low KL penalty lets the policy drift far from the base
  model and can damage answers that have nothing to do with style. Sweep it, and report
  correctness at each value.
- **One base model.** Anything observed here is about that model at that size.